# Universal fd_cache generator (Kaggle GPU)

Generate FontDiffusion handwritten-style images for **every chu-Nom character** in the NomNaTong font CJK ranges (~21,800 chars). One-time run — produces a permanent cache that any future book can reuse without re-generating on Kaggle/Colab.

**Hardware**: Kaggle **GPU T4** (Settings → Accelerator → GPU T4 x2). P100 is faster but Kaggle's PyTorch 2.10 dropped Pascal (sm_60) support, so P100 fails. Time budget on T4: **~10-12 h, may need 2 sessions**.

**Resume-able**: every 500 chars, the notebook pushes the current state to a HuggingFace Hub repo. If Kaggle resets the session, just restart the notebook — cell 5 will re-download the partial cache and skip already-generated chars.

**Output**: `mdnt571/gannhanocr-universal-fd-cache` containing 21,837 PNGs named `U+XXXX.png`. Final size ~1.5 GB.

**Style strategy**: 1 representative crop from CacThanhTruyen2 — a clean, well-scanned book — used as the style reference for all chars. This produces a consistent style across the whole cache.

## 1. Verify GPU + install deps

Check that GPU is enabled (notebook should already be set to GPU P100 in Settings).

Adds Kaggle-specific bits: secrets for `HF_TOKEN`, `tqdm` for progress.

In [ ]:
!nvidia-smi || echo '⚠️  No GPU. Settings → Accelerator → GPU T4.'

import subprocess
gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip()
print(f'GPU detected: {gpu_name}')

%pip install -q torch==2.9.1 torchvision==0.24.1 \
    --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3

%pip install -q \
    diffusers==0.36.0 \
    accelerate==1.12.0 \
    safetensors==0.7.0 \
    einops==0.8.1 \
    kornia==0.8.2 \
    info-nce-pytorch==0.1.4 \
    fonttools==4.61.1 \
    pygame==2.6.1 \
    scikit-image==0.26.0 \
    scipy==1.17.0 \
    huggingface-hub==0.36.0 \
    hf-xet==1.2.0 2>&1 | tail -3

# Gỡ peft/transformers để tránh ImportError 'is_offline_mode' khi diffusers gọi model.to()
%pip uninstall -y -q peft transformers 2>&1 | tail -2
print('✓ Removed peft/transformers (not needed by FontDiffusion).')

print('\n⚠️  If torch was downgraded, RESTART the kernel before running the next cell.\n')

import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    supported = torch.cuda.get_arch_list()
    sm = f'sm_{cap[0]}{cap[1]}'
    ok = sm in supported
    print(f'  GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
    print(f'  Compute capability: {sm}  → {"✓ supported" if ok else "✗ NOT in supported list " + str(supported)}')
    if not ok:
        print('\n🛑 GPU not compatible with this PyTorch. Switch to T4 in Settings.')

import diffusers
print(f'diffusers {diffusers.__version__}  (must be 0.36.0)')
assert diffusers.__version__ == '0.36.0', \
    f'diffusers version mismatch! Got {diffusers.__version__}, need 0.36.0. Restart kernel and re-run.'

try:
    import peft  # noqa: F401
    raise RuntimeError('peft is still installed — restart kernel and re-run this cell.')
except ImportError:
    print('✓ peft absent → diffusers will skip the broken layerwise_casting import.')


## 2. Configure HuggingFace Hub

We push the partial cache to a HF Hub repo every 500 chars so a Kaggle session reset never wipes progress. Create the repo once (any name) and add `HF_TOKEN` to Kaggle Secrets (Add-ons → Secrets).

Edit `HF_REPO` below to your repo (will be created if missing).

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, create_repo

# ════════════════════════════════════════════════════════════════════
# Your HuggingFace Hub repo for the universal fd_cache.
# This will be auto-created if it doesn't exist.
# Format: '<hf-username>/<repo-name>'
# ════════════════════════════════════════════════════════════════════
HF_REPO = 'mdnt571/gannhanocr-universal-fd-cache'
HF_REPO_TYPE = 'dataset'
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')  # add HF_TOKEN to Kaggle Add-ons → Secrets
# ════════════════════════════════════════════════════════════════════

login(token=HF_TOKEN, add_to_git_credential=False)

# Verify token + show identity
api = HfApi()
me = api.whoami()
print(f'✓ Logged in as: {me["name"]}')

expected_user = HF_REPO.split('/')[0]
if me['name'] != expected_user:
    print(f'⚠️  Token belongs to "{me["name"]}" but HF_REPO uses "{expected_user}".')
    print(f'    Either change HF_REPO to "{me["name"]}/..." or use a token from "{expected_user}".')

# Create dataset repo if missing
try:
    create_repo(repo_id=HF_REPO, repo_type=HF_REPO_TYPE, exist_ok=True, private=False)
    print(f'✓ HF repo ready: https://huggingface.co/datasets/{HF_REPO}')
except Exception as e:
    print(f'⚠️  create_repo failed: {e}')
    print('   If your token is fine-grained, ensure it has:')
    print('     • Write access to repos under your username')
    print('     • "Create new repos" enabled')
    print('   Easiest fix: create a token of type "Write" instead of "Fine-grained".')

## 3. Clone the GanNhanOCR repo (with submodule)

Pulls main repo + `font_diffusion` submodule (FontDiffusion code + fonts).

In [3]:
import os, sys, shutil
from pathlib import Path

REPO_URL = 'https://github.com/truong571/GanNhanOCR.git'
PROJECT_ROOT = Path('/kaggle/working/GanNhanOCR')

def _populated(root: Path) -> bool:
    return (root / 'font_diffusion' / 'fonts' / 'NomNaTong-Regular.ttf').exists()

if PROJECT_ROOT.exists():
    !cd {PROJECT_ROOT} && git pull --ff-only && git submodule update --init --recursive
    if not _populated(PROJECT_ROOT):
        # Stale empty submodule — wipe and re-init
        shutil.rmtree(PROJECT_ROOT / 'font_diffusion', ignore_errors=True)
        !cd {PROJECT_ROOT} && git submodule update --init --recursive
else:
    !git clone --recurse-submodules --depth 1 {REPO_URL} {PROJECT_ROOT}

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

assert _populated(PROJECT_ROOT), 'Submodule clone failed — fonts not present.'
print(f'✓ Repo at {PROJECT_ROOT}')
!ls font_diffusion/fonts/ | head -5

Cloning into '/kaggle/working/GanNhanOCR'...
remote: Enumerating objects: 123400, done.
remote: Counting objects: 100% (123400/123400), done.
remote: Compressing objects: 100% (122006/122006), done.
remote: Total 123400 (delta 1402), reused 123302 (delta 1388), pack-reused 0 (from 0)
Receiving objects: 100% (123400/123400), 319.86 MiB | 42.58 MiB/s, done.
Resolving deltas: 100% (1402/1402), done.
Updating files: 100% (123970/123970), done.
Submodule 'font_diffusion' (https://github.com/dzungphieuluuky/FontDiffusion.git) registered for path 'font_diffusion'
Cloning into '/kaggle/working/GanNhanOCR/font_diffusion'...
remote: Enumerating objects: 25434, done.        
remote: Counting objects: 100% (855/855), done.        
remote: Compressing objects: 100% (139/139), done.        
remote: Total 25434 (delta 786), reused 746 (delta 716), pack-reused 24579 (from 4)        
Receiving objects: 100% (25434/25434), 317.43 MiB | 35.92 MiB/s, done.
Resolving deltas: 100% (3928/3928), done.
Submodu

In [ ]:
# ── 3.5  Materialize medoid.png inside the cloned repo ─────────────────────
#
# medoid.png is the cosine-medoid of 120 chu-Nom crops produced by
# kaggle_diffusion/build_style_medoid.py on the Mac. It is .gitignored
# (*.png blanket rule) so `git clone` in cell 3 does NOT bring it. We embed
# the file inline as base64 so the notebook is self-contained — no Kaggle
# Dataset, no manual upload, survives session reset.
#
# To refresh after re-running build_style_medoid.py on the Mac, regenerate the
# blob with:
#   python3 -c "import base64,textwrap; print(textwrap.fill(base64.b64encode(open('kaggle_diffusion/style_references/medoid.png','rb').read()).decode(), 76))"
# and paste the output between the triple-quoted MEDOID_B64 below.

import base64

MEDOID_B64 = """
iVBORw0KGgoAAAANSUhEUgAAAHQAAABZCAIAAABgypeYAAAFd0lEQVR4Ae3BAYqsSgIAwcz7HzoX
BEFpq0e7Lf8ybyKs+DOHFX/msOLPHFb8mcOKP3NY8WcOK/7MYcWfOaz4M4cVf+aw4s8cVsyksqj4
x1gxk8qq4l9ixUwqRyp+OyvmUzlS8XtZ8QiVIxW/lBXzqQxU/FJWPELlSMWt1Ir/A1Y8QuVIxX1U
jlQ8zopHqBypuInKWxUPsuIRKkcqbqLyVsWDrHiEypGKm6i8VfEgKx6hcqTiPipjFQ+yYj6VIxW3
UhmreJAVk6m8qJhAZaziQVZMpvKiYgKVsYoHWTGTypGKCVROq5jJislUXlTcTWWvUhmrmMaKyVT2
KiZQ2ahYqRypmMaKmVRWFTOprCr2VF5UTGPFNCqriitU9irGVDYqXqgcqZjAimlUVhVjKj+pGFNZ
VYypvKi4mxXTqKwqBlROqBhQ2agYUzlScSsrplFZVbxQOa3iiMpGxQkqAxV3sGIOlVXFEZVzKsZU
VhUnqAxU3MGKCVT2KhYqF1WMqWxUvFArNlTGKr5mxa1UBiqViyreUtmo2FNZVGyoDFR8zYqbqFxX
saeyqnhLZaPiiMqiYkPlrYpPWXETlYsqXqisKsZU9iqOqCwq9lTGKj5lxXVqxZ7KORVjKouKMZW9
igGVVcWeyljFR6y4SGVRsaeyUbFQWVW8pbKoGFPZqBhQ2ah4oTJQ8RErLlJZVZyjsqgYU1lVjKls
VIyp7FWMqawqPmLFFSp7FeeoQMWAykbFmMqq4i2VFxUDKhsV11lxkcpexR1UVhVjKnsVAypHKsZU
9iqusOI6lb2KO6gsKsZU9ipA5bSKt1T2Kk6z4iMqexXfUVlVDKhsVCxUTqv4icqLinOs+IjKXsV3
VF5UbKhsVGyonFBxjsqLihOs+JTKquInKouKIypHKlYqGxUnqGxUnKNypOInVnxKZaPiLZVFxU9U
VhUrlVXFOSobFeeoHKn4iRWfUtmoeEtlUfETlUXFSmWj4hyVjYoTVMYq3rLiUyp7FWMqi4ojKlCp
rCpWKquK01Q2Kn6i8pOKMSu+oLJXMaCyqNhTGahYqawqTlPZqBhQuajiiBVfUHlRcURlUbGhMlCx
UtmoOE1lr2JD5QsVL6z4gsqLiiMqq4qFykDFSmWj4gqVr1WAypGKPSu+oHKk4oXKaRUbKquK61S+
ULFSOVKxYcUXVI5UvFA5rWJPZVFxncoJFeeovKhYWfEdlSMVGyqnVdxE5ZyKj6isKlZWfEflCxX3
UTmn4j4qi4qVFV9Tua7iPiqnVdxNBSpWVtxB5a1KZVExoLKoOE3lior5rLiJykAFqCwqBlRWFeeo
XFExnxW3UnlRASqLigGVtyqOqCwqXqhsVMxnxd1U9ipAZVExoPJWxXUqGxXzWTGBykYFqCwqBlTe
qrhOZaNiPisepLKoOKLyVsV1KhsV81nxIJVFxRGVtyquU9momM+KB6ksKo6ovFVxncpGxXxWPEhl
UfFC5YSKi1Q2Kuaz4kEqi4o9ldMqrlDZqJjPimepLCpAZaACVI5UnKayV3FEBSqVRcVHrHiWyqIC
VI5UrFQGKk5Q+ULFdVY8SOUnFS9UflIxoPKFiuuseJDKWxVjKv+pitOseJDKQMU5Kt+p2FM5p+Ic
Kx6hMlZxkco5lcpGxQuVEyrOsWI+lbGK76i8qNhQWVWMqbxVcYIVE6gsKpWxijuorCpeqKwqxlTe
qjjBivuoXFTxCJVVxVsqYxUnWHETlRMq/gsqGxVvqRypOMeKm6j8pOI/orJR8ROVFxXnWHETlUWl
sqhUFhX/EZW9isms+Aeo7FVMZsU/QGWvYjIrfjuVvYr5rPjtVPYq5rPit1PZq5jPin+AyqriEVb8
G1QWFY+w4s8c/wOooDSL4xsCTAAAAABJRU5ErkJggg==
"""

style_dir = PROJECT_ROOT / 'kaggle_diffusion' / 'style_references'
style_dir.mkdir(parents=True, exist_ok=True)
medoid_path = style_dir / 'medoid.png'
medoid_path.write_bytes(base64.b64decode(MEDOID_B64))
print(f'✓ medoid.png materialized at {medoid_path}  ({medoid_path.stat().st_size} bytes)')


## 4. Download FontDiffusion checkpoints from HF Hub

Pulls the 7 component files (unet, encoders, projections) into both DRO/ and FST/ checkpoint dirs so the loader finds them.

In [ ]:
from huggingface_hub import hf_hub_download

# Use the PRODUCTION weights at the HF repo root (fully trained, non-FST).
# The /FST/checkpoint_step_1500/ subfolder is a training snapshot at step 1500
# and produces near-noise output — do NOT use it.
# Production root only has unet/style_encoder/content_encoder; we run with
# use_fst=False (already set in core/ranking/fontdiffusion_gen.py).

FD_HF = 'dzungpham/font-diffusion-weights'
CKPT_DIR = PROJECT_ROOT / 'font_diffusion' / 'ckpt' / 'PROD'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

TO_FETCH = [
    'unet.safetensors',
    'style_encoder.safetensors',
    'content_encoder.safetensors',
]
for fn in TO_FETCH:
    dst = CKPT_DIR / fn
    if not dst.exists():
        cached = hf_hub_download(repo_id=FD_HF, filename=fn)  # ROOT, not FST/...
        shutil.copy2(cached, dst)
    print(f'  ✓ {fn}  ({dst.stat().st_size / 1024 / 1024:.1f} MB)')

# Wrapper still expects phase1_ckpt_dir for FST path; point it at the same dir
# (use_fst=False so it won\'t actually load FST modules).
FST_CKPT_DIR = CKPT_DIR

print(f'\n✓ FontDiffusion production weights ready at {CKPT_DIR}')


## 5. Resume from previous run (if any)

Pulls existing PNGs from the HF Hub repo into the local cache dir. If this is the first run, no-op.

In [5]:
from huggingface_hub import snapshot_download

CACHE_DIR = Path('/kaggle/working/_universal_fd_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Pulling existing cache from {HF_REPO} ...')
try:
    snapshot_download(
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE,
        local_dir=str(CACHE_DIR),
        allow_patterns='U+*.png',  # only PNGs, skip README etc.
    )
except Exception as e:
    print(f'  (no remote cache yet: {type(e).__name__})')

existing = sorted(CACHE_DIR.glob('U+*.png'))
print(f'✓ Resume state: {len(existing)} PNGs already on remote (will skip)')

Pulling existing cache from mdnt571/gannhanocr-universal-fd-cache ...


Fetching 0 files: 0it [00:00, ?it/s]

✓ Resume state: 0 PNGs already on remote (will skip)


## 6. Build the work list

Read `kaggle_diffusion/exports/char_universe.txt` (21,837 chars). Filter out chars already in the cache. The remaining list is what this session needs to generate.

In [6]:
universe_file = PROJECT_ROOT / 'kaggle_diffusion' / 'exports' / 'char_universe.txt'
with open(universe_file, encoding='utf-8') as f:
    all_chars = [line.strip() for line in f if line.strip()]

done_codepoints = {p.stem for p in existing}  # 'U+4E8C' etc.

todo = []
for c in all_chars:
    cp = f'U+{ord(c):04X}'
    if cp not in done_codepoints:
        todo.append(c)

print(f'Total universe:  {len(all_chars):>6,} chars')
print(f'Already done:    {len(done_codepoints):>6,}')
print(f'To generate:     {len(todo):>6,} chars')
if not todo:
    print('\n🎉 Nothing to do — cache complete!')

Total universe:  21,837 chars
Already done:         0
To generate:     21,837 chars


## 7. Load FontDiffusion pipeline (one time)

In [ ]:
import logging, warnings
logging.getLogger('src.builders.build').setLevel(logging.ERROR)
logging.getLogger('diffusers').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

from core.ranking.fontdiffusion_gen import FontDiffusionGenerator

print('Loading FontDiffusion (~30 s)...')
generator = FontDiffusionGenerator(
    ckpt_dir=str(CKPT_DIR),
    phase1_ckpt_dir=str(FST_CKPT_DIR),
    font_path=str(PROJECT_ROOT / 'font_diffusion' / 'fonts' / 'NomNaTong-Regular.ttf'),
    cache_dir=str(CACHE_DIR),
)
print('✓ Loaded')


## 7.5 Sanity test — generate 5 sample chars

Before committing to a 6-8 h run, generate 5 representative chars and verify the
output looks reasonable. This is fast (~30 s) and catches:
- Broken style image (empty / wrong format)
- FontDiffusion config mismatch
- GPU OOM at this batch size
- Per-char generator errors

If the output thumbnails look like handwritten chu-Nom (not noise / blank), proceed
to cell 8. Otherwise, fix the issue and re-run this cell.

In [ ]:
import time, logging, warnings
import numpy as np
import cv2
from PIL import Image
from IPython.display import display

logging.getLogger('src.builders.build').setLevel(logging.ERROR)
logging.getLogger('diffusers').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

STYLE_NAME_ID = 'medoid'
GUIDANCE_SCALE = 2.0
ERODE_ITERS = 2

STYLE_IMAGE_PATH = str(PROJECT_ROOT / 'kaggle_diffusion' / 'style_references' / f'{STYLE_NAME_ID}.png')
assert Path(STYLE_IMAGE_PATH).exists(), f'Style image missing: {STYLE_IMAGE_PATH}'
print(f'Style: {STYLE_IMAGE_PATH}')
display(Image.open(STYLE_IMAGE_PATH).resize((128, 128)))

try:
    generator._load_pipeline()
except Exception as e:
    print(f'✗ _load_pipeline failed: {type(e).__name__}: {e}')
    raise
generator.pipe.guidance_scale = GUIDANCE_SCALE
print(f'guidance_scale = {generator.pipe.guidance_scale}  |  erode_iters = {ERODE_ITERS}')


def erode_strokes(img_path, iters):
    try:
        arr = np.array(Image.open(img_path).convert('L'))
        if iters > 0:
            arr = cv2.dilate(arr, np.ones((3, 3), np.uint8), iterations=iters)
        Image.fromarray(arr).save(img_path)
    except Exception as e:
        print(f'  ⚠️  erode {img_path.name}: {type(e).__name__}: {e}')


TEST_CHARS = ['一', '二', '三', '人', '月']
print(f'\nGenerating {len(TEST_CHARS)} sanity-test chars: {TEST_CHARS}')

t0 = time.time()
try:
    generator.generate(TEST_CHARS, STYLE_IMAGE_PATH, style_name='_sanity')
    print(f'  Time: {(time.time()-t0):.1f}s ({(time.time()-t0)/len(TEST_CHARS):.1f}s/char)')
except Exception as e:
    print(f'  ✗ generate failed: {type(e).__name__}: {e}')
    raise

sanity_dir = CACHE_DIR / '_sanity'
for c in TEST_CHARS:
    p = sanity_dir / f'U+{ord(c):04X}.png'
    if p.exists():
        erode_strokes(p, ERODE_ITERS)

print('\nGenerated images (post-erosion):')
for c in TEST_CHARS:
    img_path = sanity_dir / f'U+{ord(c):04X}.png'
    if img_path.exists():
        print(f'  ✓ {c} → U+{ord(c):04X}.png')
        display(Image.open(img_path).resize((96, 96)))
    else:
        print(f'  ✗ {c} → not generated')

import shutil as _sh
if sanity_dir.exists():
    _sh.rmtree(sanity_dir, ignore_errors=True)
print('\n✓ Sanity test complete. Inspect images above. If they look like handwritten chu-Nom, proceed.')


## 8. Generate + resilient checkpoint upload

Loop: generate in chunks of `CHUNK`, flatten finished PNGs into the cache, then checkpoint to HF Hub with `HfApi.upload_large_folder()`. That uploader is resumable and retries transient network errors, which is safer than committing the whole cache with `upload_folder()` after every chunk.

In [ ]:
import os, time, threading, traceback, logging, warnings
import numpy as np
import cv2
from PIL import Image
from tqdm.auto import tqdm
import huggingface_hub

logging.getLogger('src.builders.build').setLevel(logging.ERROR)
logging.getLogger('diffusers').setLevel(logging.ERROR)
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')
os.environ.setdefault('HF_XET_HIGH_PERFORMANCE', '1')

CHUNK = 1000
UPLOAD_NUM_WORKERS = 4
UPLOAD_RETRIES = 4
UPLOAD_BACKOFF_SECONDS = 60
WATCHER_INTERVAL_SEC = 60           # how often the background thread scans sub/ for new PNGs
WATCHER_MIN_FILES = 200             # don't kick off a new upload until this many new files (avoid HF 429 rate limit)
FILE_STABLE_AGE_SEC = 2             # only flatten files whose mtime is older than this (avoids racing PIL writes)
STYLE_NAME_ID = 'medoid'
GUIDANCE_SCALE = 2.0
ERODE_ITERS = 2
STYLE_NAME = 'universal'
STYLE_IMAGE = str(PROJECT_ROOT / 'kaggle_diffusion' / 'style_references' / f'{STYLE_NAME_ID}.png')

assert Path(STYLE_IMAGE).exists(), f'Style image missing: {STYLE_IMAGE}'
if not hasattr(api, 'upload_large_folder'):
    raise RuntimeError(
        f'huggingface_hub {huggingface_hub.__version__} lacks upload_large_folder(). '
        'Re-run cell 1, then restart kernel.'
    )

generator._load_pipeline()
generator.pipe.guidance_scale = GUIDANCE_SCALE
print(f'STYLE = {STYLE_NAME_ID}  |  guidance_scale = {GUIDANCE_SCALE}  |  erode_iters = {ERODE_ITERS}')
print(f'huggingface_hub = {huggingface_hub.__version__}  |  CHUNK = {CHUNK}  |  watcher every {WATCHER_INTERVAL_SEC}s')


def _erode_one(p):
    try:
        arr = np.array(Image.open(p).convert('L'))
        if ERODE_ITERS > 0:
            arr = cv2.dilate(arr, np.ones((3, 3), np.uint8), iterations=ERODE_ITERS)
        Image.fromarray(arr).save(p)
    except Exception as e:
        print(f'  ⚠️  erode {p.name}: {type(e).__name__}: {e}', flush=True)


sub = CACHE_DIR / STYLE_NAME
sub.mkdir(parents=True, exist_ok=True)

_flatten_lock = threading.Lock()


def flatten(require_stable=True):
    """Move PNGs from sub/ → CACHE_DIR root.
    With require_stable=True, skip files modified within FILE_STABLE_AGE_SEC (PIL may still
    be writing). Final-flush passes False to drain everything."""
    moved = []
    now = time.time()
    with _flatten_lock:
        for p in list(sub.glob('U+*.png')):
            try:
                if require_stable and (now - p.stat().st_mtime) < FILE_STABLE_AGE_SEC:
                    continue
                dst = CACHE_DIR / p.name
                if not dst.exists():
                    _erode_one(p)
                    shutil.move(str(p), str(dst))
                    moved.append(dst)
                else:
                    p.unlink()
            except FileNotFoundError:
                continue
            except Exception as e:
                print(f'  ⚠️  flatten {p.name}: {type(e).__name__}: {e}', flush=True)
    return moved


_upload_lock = threading.Lock()
_upload_state = {'thread': None}


def _upload_blocking(label):
    n = sum(1 for _ in CACHE_DIR.glob('U+*.png'))
    if n == 0:
        print(f'  ⏫ [{label}] no PNGs to upload yet.', flush=True)
        return True
    for attempt in range(1, UPLOAD_RETRIES + 1):
        try:
            api.upload_large_folder(
                repo_id=HF_REPO,
                repo_type=HF_REPO_TYPE,
                folder_path=str(CACHE_DIR),
                allow_patterns='U+*.png',
                num_workers=UPLOAD_NUM_WORKERS,
                print_report=True,
                print_report_every=60,
            )
            print(f'  ✓ [{label}] checkpoint pushed ({n:,} PNGs).', flush=True)
            return True
        except Exception as e:
            wait = UPLOAD_BACKOFF_SECONDS * attempt
            print(f'  ⚠️  [{label}] upload {attempt}/{UPLOAD_RETRIES}: {type(e).__name__}: {e}', flush=True)
            if attempt == UPLOAD_RETRIES:
                print(f'  ⚠️  [{label}] giving up — local PNGs intact, will retry next checkpoint.', flush=True)
                return False
            time.sleep(wait)
    return False


def _upload_target(label):
    with _upload_lock:
        try:
            _upload_blocking(label)
        except Exception as e:
            print(f'  ⚠️  [{label}] background upload crashed: {type(e).__name__}: {e}', flush=True)


def maybe_upload_async(label):
    """Start an upload thread if none is running. Otherwise skip — the watcher will retry next
    interval, and chunk boundaries also call this. Non-blocking, so the watcher never stalls."""
    prev = _upload_state['thread']
    if prev is not None and prev.is_alive():
        return False
    t = threading.Thread(target=_upload_target, args=(label,), daemon=True)
    t.start()
    _upload_state['thread'] = t
    return True


def upload_wait():
    t = _upload_state['thread']
    if t is not None and t.is_alive():
        print('  ⏳ waiting for in-flight upload...', flush=True)
        t.join()


# ── Watcher: flattens new PNGs and triggers uploads while generate() is running ──
_watcher_state = {'stop': False, 'thread': None, 'pending': 0}


def _watcher_loop():
    while not _watcher_state['stop']:
        try:
            moved = flatten(require_stable=True)
            if moved:
                _watcher_state['pending'] += len(moved)
                prev = _upload_state['thread']
                idle = prev is None or not prev.is_alive()
                if idle and _watcher_state['pending'] >= WATCHER_MIN_FILES:
                    n = _watcher_state['pending']
                    _watcher_state['pending'] = 0
                    maybe_upload_async(label=f'watcher+{n}')
        except Exception as e:
            print(f'  ⚠️  watcher: {type(e).__name__}: {e}', flush=True)
        for _ in range(WATCHER_INTERVAL_SEC):
            if _watcher_state['stop']:
                return
            time.sleep(1)


def start_watcher():
    _watcher_state['stop'] = False
    _watcher_state['pending'] = 0
    t = threading.Thread(target=_watcher_loop, daemon=True)
    t.start()
    _watcher_state['thread'] = t
    print(f'  ▶  watcher started (every {WATCHER_INTERVAL_SEC}s, min {WATCHER_MIN_FILES} new files)', flush=True)


def stop_watcher():
    _watcher_state['stop'] = True
    t = _watcher_state['thread']
    if t is not None:
        t.join(timeout=WATCHER_INTERVAL_SEC + 5)
    _watcher_state['thread'] = None
    print('  ⏹  watcher stopped', flush=True)


def safe_generate(batch):
    failed = []
    try:
        generator.generate(batch, STYLE_IMAGE, style_name=STYLE_NAME)
        return failed
    except Exception as e:
        print(f'  ⚠️  batch generate failed: {type(e).__name__}: {e} — falling back to per-char', flush=True)
        torch.cuda.empty_cache()
        for c in batch:
            try:
                generator.generate([c], STYLE_IMAGE, style_name=STYLE_NAME)
            except Exception as ce:
                failed.append((c, f'{type(ce).__name__}: {ce}'))
                torch.cuda.empty_cache()
        if failed:
            print(f'  ⚠️  {len(failed)} chars failed in this chunk (will retry next session).', flush=True)
        return failed


if todo:
    t0 = time.time()
    pbar = tqdm(total=len(todo), desc='Generate', unit='char', dynamic_ncols=True)
    all_failed = []
    start_watcher()
    try:
        for i in range(0, len(todo), CHUNK):
            batch = todo[i:i + CHUNK]
            missing_batch = [c for c in batch if not (CACHE_DIR / f'U+{ord(c):04X}.png').exists()]

            if missing_batch:
                all_failed.extend(safe_generate(missing_batch))
            else:
                print(f'  chunk {i}–{i+len(batch)}: already present locally; checkpointing only', flush=True)

            # Safety net at chunk boundary — watcher already moved most files, but drain leftovers.
            moved = flatten(require_stable=True)
            if moved:
                _watcher_state['pending'] += len(moved)
                print(f'  flattened {len(moved):,} extra PNGs after chunk', flush=True)

            pbar.update(len(batch))
            if maybe_upload_async(label=f'{min(i + CHUNK, len(todo)):,}/{len(todo):,}'):
                _watcher_state['pending'] = 0
            torch.cuda.empty_cache()
    except KeyboardInterrupt:
        print('\n⚠️  interrupted — letting in-flight upload finish so progress is saved.')
    except Exception as e:
        print(f'\n⚠️  loop error: {type(e).__name__}: {e}')
        traceback.print_exc()
    finally:
        pbar.close()
        stop_watcher()
        # Final flush: generation is done, so drain everything (no stable-age skip needed).
        leftover = flatten(require_stable=False)
        if leftover:
            print(f'  flattened {len(leftover):,} leftover PNGs at end', flush=True)
        upload_wait()
        _upload_blocking(label='final')
        if all_failed:
            print(f'\n⚠️  {len(all_failed)} chars failed total (showing first 10):')
            for c, err in all_failed[:10]:
                print(f'    {c} (U+{ord(c):04X}): {err}')
        print(f'\n✓ done in {(time.time()-t0)/60:.1f} min')
else:
    print('Nothing to generate.')


## 9. Final state + verification

After generation finishes (or session resets), confirm cache size matches universe.

In [ ]:
final = sorted(CACHE_DIR.glob('U+*.png'))
size_mb = sum(p.stat().st_size for p in final) / 1024 / 1024
print(f'Final cache: {len(final):,} PNGs / {len(all_chars):,} target')
print(f'Total size: {size_mb:.0f} MB')
print(f'Coverage: {len(final)/len(all_chars)*100:.1f}%')

if len(final) < len(all_chars):
    missing_codepoints = set(f'U+{ord(c):04X}' for c in all_chars) - {p.stem for p in final}
    print(f'\n{len(missing_codepoints)} chars still missing.')
    print('Restart this notebook to resume — cell 5 will pull state and continue.')

## 10. Use the cache from your Mac

Once 100% complete, on the Mac side:

```sh
cd /path/to/GanNhanOCR
huggingface-cli download mdnt571/gannhanocr-universal-fd-cache \
  --repo-type=dataset \
  --local-dir prepared/_universal_fd_cache/

./run_pipeline.sh --step 3 && ./run_pipeline.sh --step 4
```

Step 3 will use the universal cache for any chu-Nom char in any book — no more Colab/Kaggle generation runs needed.